# Company CNC Dataset Training

This notebook reproduces the offline training workflow for the curated Nakamura 2 company dataset added to the repository.

It is intentionally separate from the live edge demo because the company dataset uses a different feature schema than the current simulator and edge inference service.

In [ ]:
from industrial_mlops.company_dataset import (
    COMPANY_FEATURE_COLUMNS,
    COMPANY_TARGET_COLUMN,
    load_company_reference_dataset,
    load_company_reference_metadata,
)
from industrial_mlops.db import load_company_reference_training_dataset, seed_company_reference_data_if_empty
from industrial_mlops.ml import evaluate_model, train_model
from industrial_mlops.registry import train_and_register_company_reference

import pandas as pd

In [ ]:
metadata = load_company_reference_metadata()
frame = load_company_reference_dataset()

print(metadata['label_definition'])
print(metadata['curated_dataset'])
display(frame.head())
display(frame.groupby(['split', 'actual_breakage']).size().unstack(fill_value=0))

## Optional Timescale Check

If the Docker stack is running, the bootstrap service will also seed the curated dataset into `company_reference_events`.

In [ ]:
try:
    seeded = seed_company_reference_data_if_empty()
    db_frame = load_company_reference_training_dataset()
    print({'seeded_or_existing_rows': seeded, 'db_rows': len(db_frame)})
except Exception as exc:
    print(f'Timescale check skipped: {exc}')

## Offline Training and Hold-Out Evaluation

The curated dataset already contains chronological `train`, `validation` and `test` splits.

For this notebook, `train + validation` are merged for fitting, and `test` is kept as a final hold-out set.

In [ ]:
training_frame = frame[frame['split'].isin(['train', 'validation'])].reset_index(drop=True)
test_frame = frame[frame['split'] == 'test'].reset_index(drop=True)

result = train_model(
    training_frame,
    feature_columns=COMPANY_FEATURE_COLUMNS,
    target_column=COMPANY_TARGET_COLUMN,
    random_state=42,
)
test_metrics = evaluate_model(
    result.model,
    test_frame,
    feature_columns=COMPANY_FEATURE_COLUMNS,
    target_column=COMPANY_TARGET_COLUMN,
)

print('Internal training split metrics:', result.metrics)
print('Chronological hold-out metrics:', test_metrics)

## Optional MLflow Registration

Set `register_to_mlflow = True` to create a new version under the company reference model name `cnc_company_reference_classifier`.

The automatic bootstrap already creates a baseline version if none exists. This cell is useful when you want to log a new experiment from the notebook.

In [ ]:
register_to_mlflow = False

if register_to_mlflow:
    registration = train_and_register_company_reference(
        training_frame,
        reason='notebook-company-reference',
        run_name='company-reference-notebook',
    )
    display(registration)
else:
    print('Set register_to_mlflow = True to register a new MLflow version.')